# 00 · Verify the bundle and the environment (minutes)

Run first on any machine. Nothing is mined. Checks:

1. every shipped input (data, saved results, library source, tool sources, harness)
   against `SHA256SUMS` (files under `data/` when present: the repository copy has no data);
2. Python and package versions against `env/requirements.txt`;
3. the installed `fastapriori` (must be 0.2.0 with the new kernels);
4. on Linux: `/usr/bin/time`, the Borgelt binaries and LCM built by `env/setup_vm.sh`.

The last cell says which notebooks this machine can run.

In [1]:
# ---- bundle configuration (same cell in every notebook) ----
import os, sys, json, time, hashlib
from pathlib import Path
import numpy as np
import pandas as pd

BUNDLE = Path.cwd().resolve().parent          # notebooks run from <bundle>/notebooks
assert (BUNDLE / "bench" / "baseline_runners.py").exists(), f"start Jupyter in <bundle>/notebooks (cwd={Path.cwd()})"
sys.path.insert(0, str(BUNDLE / "bench"))
DATA = Path(os.environ.get("FA_DATA_DIR", BUNDLE / "data"))
RESULTS = BUNDLE / "results"
DERIVED = RESULTS / "derived"                       # regenerated statistics, tables and figures

V010 = "campaign_2026-05"                           # the v0.1.0.post1 GCE campaign
V010_DIR = RESULTS / V010
CAMPAIGN = os.environ.get("FA_CAMPAIGN", V010) # set FA_CAMPAIGN=<id> to analyse a new run
CAMPAIGN_DIR = V010_DIR if CAMPAIGN == V010 else RESULTS / "runs" / CAMPAIGN
AGG_CSV = (V010_DIR / "results_rigorous_20260523_115442.csv" if CAMPAIGN == V010
           else CAMPAIGN_DIR / f"results_rigorous_{CAMPAIGN}.csv")
LCM_CSV = V010_DIR / "lcm_declat_results_20260523_055523.csv"
COST_CSV = (DERIVED / "cost_stats.csv") if (DERIVED / "cost_stats.csv").exists() else V010_DIR / "cost_stats_20261022.csv"

def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def rel(p):
    try:
        return Path(p).resolve().relative_to(BUNDLE).as_posix()
    except ValueError:
        return str(p)

pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)
print(f"campaign : {CAMPAIGN}\nagg csv  : {rel(AGG_CSV)}")

campaign : campaign_2026-05
agg csv  : results/campaign_2026-05/results_rigorous_20260523_115442.csv


In [2]:
bad, n, absent = [], 0, 0
for line in (BUNDLE / "SHA256SUMS").read_text(encoding="utf-8").splitlines():
    digest, name = line.split("  ", 1)
    p = BUNDLE / name
    n += 1
    if not p.exists():
        if name.startswith("data/"):       # the repository copy has no data/
            absent += 1
        else:
            bad.append((name, "missing"))
    elif sha256(p) != digest:
        bad.append((name, "changed"))
print(f"{n - len(bad) - absent}/{n} shipped files verified" + (f", {absent} data files absent" if absent else ""))
assert not bad, bad[:20]

164/164 shipped files verified


In [3]:
import platform, importlib.metadata as md, re
print("python", platform.python_version(), "|", platform.platform())
rows = []
for line in (BUNDLE / "env" / "requirements.txt").read_text().splitlines():
    m = re.match(r"^([A-Za-z0-9_.\-]+)==([^\s#]+)", line.strip())
    if not m: continue
    name, want = m.groups()
    try: have = md.version(name)
    except md.PackageNotFoundError: have = None
    rows.append(dict(package=name, required=want, installed=have, ok=have == want))
pkgs = pd.DataFrame(rows)
display(pkgs[~pkgs.ok] if (~pkgs.ok).any() else "all pinned packages installed at the pinned version")

python 3.12.3 | Linux-6.18.54.1-microsoft-standard-WSL2-x86_64-with-glibc2.39


'all pinned packages installed at the pinned version'

In [4]:
status = {}
try:
    import fastapriori._fastapriori_rs as rs
    v = md.version("fastapriori")
    new = all(hasattr(rs, f) for f in ("rust_compute_pairs_v2", "rust_eclat_pipeline_seeded"))
    status["fastapriori"] = f"{v} ({'with' if new else 'WITHOUT'} 0.2.0 kernels) at {rs.__file__}"
    status["lib_ok"] = v == "0.2.0" and new
except Exception as e:
    status["fastapriori"] = f"not importable: {e!r}"; status["lib_ok"] = False
linux = sys.platform.startswith("linux")
tools = BUNDLE / "tools" / "bin"
status["linux"] = linux
status["/usr/bin/time"] = Path("/usr/bin/time").exists()
for t in ("apriori", "eclat", "fpgrowth", "lcm"):
    status[f"tools/bin/{t}"] = (tools / t).exists()
status

{'fastapriori': '0.2.0 (with 0.2.0 kernels) at /home/user/.venvs/fastapriori/lib/python3.12/site-packages/fastapriori/_fastapriori_rs.cpython-312-x86_64-linux-gnu.so',
 'lib_ok': True,
 'linux': True,
 '/usr/bin/time': True,
 'tools/bin/apriori': False,
 'tools/bin/eclat': False,
 'tools/bin/fpgrowth': False,
 'tools/bin/lcm': False}

In [5]:
can_mine = status["lib_ok"] and linux and status["/usr/bin/time"] and all(status[f"tools/bin/{t}"] for t in ("apriori", "eclat", "fpgrowth"))
plan = pd.DataFrame([
    ("01 synthetic data (verify)", True), ("02 v0.1.0.post1 grid (replay)", True),
    ("02 v0.1.0.post1 grid (smoke/run)", can_mine), ("03 aggregate", True), ("04 cost statistics", True),
    ("05 tables & figures", True), ("06 regime & ML", True), ("07 cost-model correlation", True),
    ("10, 13-15, 19 v0.2.0 experiments", can_mine and status["tools/bin/lcm"]),
    ("13-15, 19 saved results (FA_MODE=replay)", True),
], columns=["notebook", "can run here"])
plan

,notebook,can run here
0,01 synthetic data (verify),True
1,02 v0.1.0.post1 grid (replay),True
2,02 v0.1.0.post1 grid (smoke/run),False
3,03 aggregate,True
4,04 cost statistics,True
5,05 tables & figures,True
6,06 regime & ML,True
7,07 cost-model correlation,True
8,"10, 13-15, 19 v0.2.0 experiments",False
9,"13-15, 19 saved results (FA_MODE=replay)",True
